# Interim: report figures

Renders the four figures referenced from `internship_report.tex`
into `CourseworkTemplateEng/graphics/`:

1. `pipeline.pdf`  -- end-to-end architecture diagram (fig:pipeline).
2. `pareto.pdf`    -- accuracy vs. log-params scatter (fig:pareto).
3. `f3_gates.pdf`  -- per-task gate histograms from F3 (fig:gates).
4. `f4_xattn.pdf`  -- cross-modal attention map from F4 (fig:xattn).

Runs top-to-bottom; each cell is idempotent and overwrites its own
PDF on re-run. Checkpoints expected under
`results/interim/<variant>/best.pt`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("cwd:", Path.cwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

GRAPHICS = ROOT.parent / "CourseworkTemplateEng" / "graphics"
# If the notebook was launched from inside thesis-code, the report folder
# is a sibling of ROOT; otherwise try the repo root layout.
if not GRAPHICS.parent.is_dir():
    GRAPHICS = ROOT.parent.parent / "CourseworkTemplateEng" / "graphics"
GRAPHICS.mkdir(parents=True, exist_ok=True)
print("graphics dir:", GRAPHICS)

cwd: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code
graphics dir: C:\Users\Andrey Lyaschenko\Documents\vkr\CourseworkTemplateEng\graphics


## 1. Pipeline diagram (`fig:pipeline`)

Three-column architecture diagram drawn with matplotlib primitives --
no external tools required. Left column: visual branch. Right column:
audio branch. Center: fusion + three heads + smoothing.

In [ ]:
def _box(ax, x, y, w, h, text, face="#eef3fb", edge="#3c5a99", fontsize=9):
    patch = FancyBboxPatch((x, y), w, h,
                           boxstyle="round,pad=0.02,rounding_size=0.05",
                           linewidth=1.0, edgecolor=edge, facecolor=face)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            fontsize=fontsize)

def _arrow(ax, x0, y0, x1, y1, style="-|>"):
    arr = FancyArrowPatch((x0, y0), (x1, y1), arrowstyle=style,
                          mutation_scale=12, linewidth=1.0, color="#333")
    ax.add_patch(arr)

fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.set_aspect("equal")
ax.axis("off")

# --- Visual branch (left column) --------------------------------------
_box(ax, 0.2, 5.5, 2.4, 0.9, "Video frames\n(112x112 RGB)")
_box(ax, 0.2, 4.1, 2.4, 0.9, "Face detection\n& alignment")
_box(ax, 0.2, 2.7, 2.4, 0.9, "Frozen visual backbone\n(EfficientNet-B0 / MBF)",
     face="#fff4e6", edge="#d08100")
_box(ax, 0.2, 1.3, 2.4, 0.9, "$(x_v, s_v)$\n1280+10 or 512+10")

# --- Audio branch (right column) -------------------------------------
_box(ax, 9.4, 5.5, 2.4, 0.9, "Audio waveform\n(16 kHz mono)")
_box(ax, 9.4, 4.1, 2.4, 0.9, "Frame-align to video\n(scipy resample)")
_box(ax, 9.4, 2.7, 2.4, 0.9, "Frozen speech encoder\n(HuBERT large / wav2vec2)",
     face="#fff4e6", edge="#d08100")
_box(ax, 9.4, 1.3, 2.4, 0.9, "$x_a$\n1024 or 768")

# --- Fusion + heads (center column) ----------------------------------
_box(ax, 4.2, 3.4, 3.6, 1.1,
     "Fusion module — 9 variants\n"
     "Late-blend, Concat-MLP, Scalar-blend, Task-gate,\n"
     "Cross-Attn, LMF, Dirichlet, IACA-gate, MBT",
     face="#e8f5e8", edge="#2e7d32", fontsize=7.5)
_box(ax, 3.4, 1.7, 1.5, 0.8, "EXPR head\nmacro-$F_1$")
_box(ax, 5.25, 1.7, 1.5, 0.8, "VA head\nCCC")
_box(ax, 7.1, 1.7, 1.5, 0.8, "AU head\nmacro-$F_1$")
_box(ax, 4.2, 0.2, 3.6, 0.9, "Per-video Gaussian smoothing\n(EXPR probs + VA regressions)",
     face="#f3e8fd", edge="#5e35b1")

# --- Arrows (vertical, within each column) ---------------------------
for x in (1.4, 10.6):
    _arrow(ax, x, 5.5, x, 5.0)
    _arrow(ax, x, 4.1, x, 3.6)
    _arrow(ax, x, 2.7, x, 2.2)

# --- Arrows into fusion ----------------------------------------------
_arrow(ax, 2.6, 1.75, 4.2, 3.7)
_arrow(ax, 9.4, 1.75, 7.8, 3.7)

# --- Arrows from fusion to heads -------------------------------------
_arrow(ax, 5.0, 3.4, 4.15, 2.5)
_arrow(ax, 6.0, 3.4, 6.0, 2.5)
_arrow(ax, 7.0, 3.4, 7.85, 2.5)

# --- Arrows from heads to smoothing ---------------------------------
for hx in (4.15, 6.0, 7.85):
    _arrow(ax, hx, 1.7, hx, 1.1)

# --- Legend ----------------------------------------------------------
ax.text(6.0, 6.6, "Frozen backbone", ha="center", fontsize=10,
        style="italic", color="#d08100")
ax.text(1.4, 6.7, "Visual branch", ha="center", fontsize=11, weight="bold")
ax.text(10.6, 6.7, "Audio branch", ha="center", fontsize=11, weight="bold")

out = GRAPHICS / "pipeline.pdf"
fig.savefig(out, bbox_inches="tight", pad_inches=0.05)
plt.close(fig)
print("wrote", out)

## 2. Pareto scatter (`fig:pareto`)

Reads the main CREMA-D leaderboard from `summary_crema.md` and plots
$P_{\mathrm{MTL}}'$ vs. log trainable params, one point per variant.
The two unimodal baselines anchor the lower-left; F4 sits at the
upper-right.

In [3]:
summary = ROOT / "results" / "interim" / "summary_crema.md"
# Parse the markdown table into a dataframe. Skip the header separator
# line (|:---|...) that pandas.read_table does not handle natively.
lines = [ln for ln in summary.read_text(encoding="utf-8").splitlines()
         if ln.startswith("|") and "---" not in ln]
rows = [[c.strip() for c in ln.strip("|").split("|")] for ln in lines]
df = pd.DataFrame(rows[1:], columns=rows[0])
for col in ("P_MTL@0.5", "trainable_params"):
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["P_MTL@0.5", "trainable_params"]).reset_index(drop=True)

COLOURS = {
    "visual_only": "#888888", "audio_only": "#888888",
    "f1_concat": "#1f77b4", "f2_blend": "#ff7f0e",
    "f3_gate": "#2ca02c",   "f4_xattn": "#d62728",
    "f5_lmf": "#9467bd",
}

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for _, row in df.iterrows():
    c = COLOURS.get(row["variant"], "#333333")
    ax.scatter(row["trainable_params"], row["P_MTL@0.5"],
               s=80, color=c, edgecolor="black", linewidth=0.5, zorder=3)
    ax.annotate(row["variant"],
                (row["trainable_params"], row["P_MTL@0.5"]),
                xytext=(6, 3), textcoords="offset points", fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("Trainable parameters (log scale)")
ax.set_ylabel(r"$P_{\mathrm{MTL}}'$ = CCC$_V$ + CCC$_A$ + $F_1^{\mathrm{EXPR}}$")
ax.grid(True, which="both", alpha=0.25, linestyle=":")
ax.set_title("CREMA-D: accuracy vs. parameter cost")

out = GRAPHICS / "pareto.pdf"
fig.savefig(out, bbox_inches="tight", pad_inches=0.05)
plt.close(fig)
print("wrote", out, "  (n=", len(df), "points)")

'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


wrote C:\Users\Andrey Lyaschenko\Documents\vkr\CourseworkTemplateEng\graphics\pareto.pdf   (n= 7 points)


## 3. F3 gate histograms (`fig:gates`)

Loads `crema_f3_gate/best.pt`, runs inference on the val split, and
plots the distribution of the sigmoid gate $\alpha_t$ for each of the
four tasks {EXPR, V, A, AU}. $\alpha_t = 1$ means the visual branch
wins; $\alpha_t = 0$ means the audio branch wins.

In [4]:
import torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

from src.datasets.bimodal import BimodalFrameDataset
from src.fusion.base import FusionConfig
from src.train_fusion import build_model

cfg = OmegaConf.load(ROOT / "configs" / "interim" / "crema_f3_gate.yaml")
ck = torch.load(ROOT / "results" / "interim" / "crema_f3_gate" / "best.pt",
                map_location="cpu", weights_only=False)
fus_cfg = FusionConfig(**ck["fus_cfg"])
model = build_model(ck["variant"], fus_cfg)
model.load_state_dict(ck["state_dict"])
device = cfg.train.device if torch.cuda.is_available() else "cpu"
model.to(device).eval()

ds = BimodalFrameDataset(
    annotations_file=cfg.data.val_annotations,
    visual_cache_dir=cfg.visual.features_cache,
    audio_cache_dir=cfg.audio.aligned_dir,
)
loader = DataLoader(ds, batch_size=256, shuffle=False)

alphas = []
with torch.no_grad():
    for batch in loader:
        model(batch["v_feat"].to(device),
              batch["v_scores"].to(device),
              batch["a_feat"].to(device))
        alphas.append(model.last_alpha().cpu().numpy())
alphas = np.concatenate(alphas, axis=0)   # (N, 4) over {EXPR, V, A, AU}
print("alpha tensor:", alphas.shape, "  mean:", alphas.mean(0).round(3))

# AU loss weight is zero for CREMA-D (no AU labels), so the AU gate is
# not meaningful -- drop it.
TASKS = [("EXPR", 0, "#1f77b4"), ("V", 1, "#d62728"), ("A", 2, "#2ca02c")]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for name, idx, colour in TASKS:
    ax.hist(alphas[:, idx], bins=40, range=(0, 1),
            alpha=0.55, color=colour, label=name,
            edgecolor="white", linewidth=0.4)
ax.axvline(0.5, color="grey", linestyle=":", linewidth=1.0)
ax.set_xlabel(r"$\alpha_t$  (1 = visual wins, 0 = audio wins)")
ax.set_ylabel("Frames on val split")
ax.set_title("F3 per-task gate distribution, CREMA-D val")
ax.legend(title="task", frameon=False, loc="upper center")
ax.grid(True, alpha=0.25, linestyle=":")

out = GRAPHICS / "f3_gates.pdf"
fig.savefig(out, bbox_inches="tight", pad_inches=0.05)
plt.close(fig)
print("wrote", out)

alpha tensor: (14988, 4)   mean: [0.759 0.244 0.454 0.313]
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\CourseworkTemplateEng\graphics\f3_gates.pdf


## 4. F4 cross-modal attention (`fig:xattn`)

Monkey-patches the `MultiheadAttention` modules inside F4's two
cross-attention blocks to capture weights. Picks one illustrative
windowed frame from the val split (high-arousal anger clip),
runs inference, and plots the $T \times T$ attention heatmap for
both directions (Q=visual, K=audio and vice versa), averaged over
heads.

In [5]:
from src.datasets.bimodal import BimodalWindowDataset

cfg = OmegaConf.load(ROOT / "configs" / "interim" / "crema_f4_xattn.yaml")
ck = torch.load(ROOT / "results" / "interim" / "crema_f4_xattn" / "best.pt",
                map_location="cpu", weights_only=False)
fus_cfg = FusionConfig(**ck["fus_cfg"])
model = build_model(ck["variant"], fus_cfg)
model.load_state_dict(ck["state_dict"])
device = cfg.train.device if torch.cuda.is_available() else "cpu"
model.to(device).eval()

# Capture attention weights from both directions.
captured = {}
def _wrap_mha(mha_module, key):
    orig_forward = mha_module.forward
    def wrapped(q, k, v, **kwargs):
        kwargs["need_weights"] = True
        kwargs["average_attn_weights"] = True
        out, wts = orig_forward(q, k, v, **kwargs)
        captured[key] = wts.detach().cpu()
        return out, wts
    mha_module.forward = wrapped
_wrap_mha(model.block_v.attn, "v_attends_a")
_wrap_mha(model.block_a.attn, "a_attends_v")

window = int(cfg.fusion.get("window", 5))
ds = BimodalWindowDataset(
    annotations_file=cfg.data.val_annotations,
    visual_cache_dir=cfg.visual.features_cache,
    audio_cache_dir=cfg.audio.aligned_dir,
    window=window,
)

# Pick the sample with strongest non-neutral EXPR signal (|VA| large).
T = 2 * window + 1
best_idx, best_score = 0, -1.0
for i in range(min(len(ds), 2000)):
    s = ds[i]
    score = float(abs(s["y_va"][0]) + abs(s["y_va"][1]))
    if score > best_score:
        best_score, best_idx = score, i
sample = ds[best_idx]
print("picked sample", best_idx, "from", sample.get("videoname"),
      " VA=", tuple(sample["y_va"].tolist()))

with torch.no_grad():
    model(sample["v_feat"].unsqueeze(0).to(device),
          sample["v_scores"].unsqueeze(0).to(device),
          sample["a_feat"].unsqueeze(0).to(device))

fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0), constrained_layout=True)
for ax, key, title in zip(axes,
                          ("v_attends_a", "a_attends_v"),
                          ("Q = visual   |   K,V = audio",
                           "Q = audio    |   K,V = visual")):
    wts = captured[key][0].numpy()   # (T, T) after averaging over heads
    im = ax.imshow(wts, cmap="viridis", aspect="equal",
                   extent=[-window - 0.5, window + 0.5,
                           window + 0.5, -window - 0.5])
    ax.axhline(0, color="white", linewidth=0.5, linestyle=":")
    ax.axvline(0, color="white", linewidth=0.5, linestyle=":")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Key offset (frames)")
    ax.set_ylabel("Query offset (frames)")
    plt.colorbar(im, ax=ax, shrink=0.85, label="attention weight")
fig.suptitle(f"F4 cross-modal attention -- sample {best_idx}", fontsize=11)

out = GRAPHICS / "f4_xattn.pdf"
fig.savefig(out, bbox_inches="tight", pad_inches=0.05)
plt.close(fig)
print("wrote", out)

picked sample 26 from None  VA= (-0.6000000238418579, 0.800000011920929)
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\CourseworkTemplateEng\graphics\f4_xattn.pdf


## 5. Dataset examples gallery (`fig:dataset_examples`)

Cropped face frames from Aff-Wild2 / CREMA-D / RAVDESS validation
splits with ground-truth and predicted EXPR labels printed next
to each crop. Predictions come from in-domain checkpoints:
Cross-Attn for Aff-Wild2 (seed 42, the headline bimodal winner)
and CREMA-D, Concat-MLP for RAVDESS. Aff-Wild2 rows also show the
ground-truth valence/arousal; CREMA-D / RAVDESS omit them (those
are circumplex-anchored and clip-constant, see §3.6).
Frames are selected to cover four distinct EXPR classes per
dataset (with a fallback if fewer than four are present in the
permuted indices).

Image roots assumed at `<repo>/competition-data/cropped_aligned/`
(Aff-Wild2), `<repo>/thesis-code/data/crema_d/cropped_aligned/`,
`<repo>/thesis-code/data/ravdess/cropped_aligned/`.

In [7]:
import torch
from omegaconf import OmegaConf
from PIL import Image

from src.datasets.bimodal import BimodalFrameDataset
from src.fusion.base import FusionConfig
from src.train_fusion import build_model

EXPR_NAMES = ["Neutral", "Anger", "Disgust", "Fear",
              "Happiness", "Sadness", "Surprise", "Other"]

# CREMA-D filename format: <actor>_<sentence>_<emotion>_<intensity>
# 12 canonical sentences keyed by the 3-letter code.
CREMAD_SENTENCES = {
    "IEO": "It's eleven o'clock",
    "TIE": "That is exactly what happened",
    "IOM": "I'm on my way to the meeting",
    "IWW": "I wonder what this is about",
    "TAI": "The airplane is almost full",
    "MTI": "Maybe tomorrow it will be cold",
    "IWL": "I would like a new alarm clock",
    "ITH": "I think I have a doctor's appointment",
    "DFA": "Don't forget a jacket",
    "ITS": "I think I've seen this before",
    "TSI": "The surface is slick",
    "WSI": "We'll stop in a couple of minutes",
}

# RAVDESS filename format: 01-01-01-01-01-01-01.
# Field 4 (0-indexed) encodes the statement.
RAVDESS_STATEMENTS = {
    "01": "Kids are talking by the door",
    "02": "Dogs are sitting by the door",
}

def caption_cremad(videoname):
    parts = videoname.split("_")
    return CREMAD_SENTENCES.get(parts[1]) if len(parts) >= 2 else None

def caption_ravdess(videoname):
    parts = videoname.split("-")
    return RAVDESS_STATEMENTS.get(parts[4]) if len(parts) >= 5 else None

# (label, prediction-source tag, config, checkpoint, image root,
#  show_va_in_caption, caption_fn(videoname) -> str | None)
DATASETS = [
    ("Aff-Wild2", "Cross-Attn",
     ROOT / "configs" / "stage3_f4_xattn.yaml",
     ROOT / "results" / "stage3_f4_xattn" / "best.pt",
     ROOT.parent / "competition-data" / "cropped_aligned",
     True, lambda vn: None),
    ("CREMA-D", "Cross-Attn (in-domain)",
     ROOT / "configs" / "interim" / "crema_f4_xattn.yaml",
     ROOT / "results" / "interim" / "crema_f4_xattn" / "best.pt",
     ROOT / "data" / "crema_d" / "cropped_aligned",
     False, caption_cremad),
    ("RAVDESS", "Concat-MLP (in-domain)",
     ROOT / "configs" / "interim" / "ravdess_f1_concat.yaml",
     ROOT / "results" / "interim" / "ravdess_f1_concat" / "best.pt",
     ROOT / "data" / "ravdess" / "cropped_aligned",
     False, caption_ravdess),
]
N_PER_ROW = 4
device = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(seed=42)

fig, axes = plt.subplots(len(DATASETS), N_PER_ROW, figsize=(13, 10.5))

for row, (name, tag, cfg_path, ckpt_path, img_root, show_va, caption_fn) in enumerate(DATASETS):
    cfg = OmegaConf.load(cfg_path)
    ck  = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    fus_cfg = FusionConfig(**ck["fus_cfg"])
    model = build_model(ck["variant"], fus_cfg)
    model.load_state_dict(ck["state_dict"])
    model.to(device).eval()

    ds = BimodalFrameDataset(
        annotations_file=cfg.data.val_annotations,
        visual_cache_dir=cfg.visual.features_cache,
        audio_cache_dir=cfg.audio.aligned_dir,
    )

    # Pick N_PER_ROW samples covering diverse EXPR classes where possible.
    picks = []
    seen_classes = set()
    indices = rng.permutation(len(ds))
    for idx in indices:
        idx = int(idx)
        y = int(ds.anno.df.iloc[idx]["expr"])
        if y < 0 or y in seen_classes:
            continue
        seen_classes.add(y)
        picks.append(idx)
        if len(picks) >= N_PER_ROW:
            break
    for idx in indices:
        if len(picks) >= N_PER_ROW:
            break
        idx = int(idx)
        if idx in picks:
            continue
        if int(ds.anno.df.iloc[idx]["expr"]) < 0:
            continue
        picks.append(idx)

    with torch.no_grad():
        for col, idx in enumerate(picks):
            sample = ds[idx]
            v  = sample["v_feat"].unsqueeze(0).to(device)
            sv = sample["v_scores"].unsqueeze(0).to(device)
            a  = sample["a_feat"].unsqueeze(0).to(device)
            out = model(v, sv, a)
            if isinstance(out, dict):
                logits = out["expr"]
            elif isinstance(out, (tuple, list)):
                logits = out[0]
            else:
                logits = out
            pred = int(logits.argmax(dim=-1).cpu().item())

            anno_row = ds.anno.df.iloc[idx]
            gt = int(anno_row["expr"])
            v_gt = float(anno_row["valence"])
            a_gt = float(anno_row["arousal"])
            relpath = str(anno_row["image_path"])
            videoname = relpath.split("/")[0]

            ax = axes[row, col]
            ax.set_xticks([]); ax.set_yticks([])
            jpeg = img_root / relpath
            if jpeg.exists():
                ax.imshow(Image.open(jpeg))
            else:
                ax.text(0.5, 0.5, f"missing:\n{relpath}",
                        ha="center", va="center",
                        fontsize=7, transform=ax.transAxes)

            mark = "✓" if pred == gt else "✗"
            title_lines = [f"GT: {EXPR_NAMES[gt]}",
                           f"Pred: {EXPR_NAMES[pred]}  {mark}"]
            if show_va and -1.0 <= v_gt <= 1.0 and -1.0 <= a_gt <= 1.0:
                title_lines.append(f"(V={v_gt:+.2f}, A={a_gt:+.2f})")
            sentence = caption_fn(videoname)
            if sentence:
                title_lines.append(f"“{sentence}”")
            ax.set_title("\n".join(title_lines), fontsize=7.5)

    axes[row, 0].set_ylabel(f"{name}\n[{tag}]", fontsize=10,
                            rotation=90, labelpad=8)

fig.suptitle("Cropped face frames per dataset, with ground-truth and predicted EXPR labels"
             "  (audio features: HuBERT-large)",
             fontsize=11, y=1.0)
fig.tight_layout()
out_path = GRAPHICS / "dataset_examples.pdf"
fig.savefig(out_path, bbox_inches="tight", pad_inches=0.05)
plt.close(fig)
print("wrote", out_path)

C:\Users\Andrey Lyaschenko\AppData\Local\Temp\ipykernel_13224\2889566840.py:154: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) DejaVu Serif.
  fig.tight_layout()
C:\Users\Andrey Lyaschenko\AppData\Local\Temp\ipykernel_13224\2889566840.py:154: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) DejaVu Serif.
  fig.tight_layout()
C:\Users\Andrey Lyaschenko\AppData\Local\Temp\ipykernel_13224\2889566840.py:156: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) DejaVu Serif.
  fig.savefig(out_path, bbox_inches="tight", pad_inches=0.05)
C:\Users\Andrey Lyaschenko\AppData\Local\Temp\ipykernel_13224\2889566840.py:156: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) DejaVu Serif.
  fig.savefig(out_path, bbox_inches="tight", pad_inches=0.05)


wrote C:\Users\Andrey Lyaschenko\Documents\CourseworkTemplateEng\graphics\dataset_examples.pdf
